This is a notebook to visualize different properties of the dataset as we go about curating. Here, we do some processing on ProtParam properties calculated using [BioPython's ProteinAnalysis tool](https://biopython.org/docs/1.76/api/Bio.SeqUtils.ProtParam.html).
Properties we care about:
- **Basic statistics** What is the percent of each kind of amino acid in the sequence?
- **Secondary structure statistics** What is the frequency of amino acids that tend to be in Helix, Turn, or Sheet, respectively. Amino acids in helix: V, I, Y, F, W, L. Amino acids in Turn: N, P, G, S. Amino acids in sheet: E, M, A, L.
- **Charge** Isoelectric point is the pH at which the protein has no net electrical charge, which can be a useful biochemical proxy for solubility, protein purification. We also want to know the charge at certain pH values that are relevant to specific cellular compartments, since some proteins are active at an optimal pH, when they're located in the right compartment.
- **Hydrophobicity** Gravy score is a proxy for how greasy a protein is, which can affect its solubility and what compartment it's found in.
- **Instability** We can calculate the tendency of a protein to be disordered or unstable, which can help understand its half-life / turnover.

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import polars as pl
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

from project.utils.strs import SEED, data_dir

from project.utils.splitting import hierarchical_clustered_split

from project.utils.functions import check_correlation_plotly


In [ ]:
subset_dir = data_dir / 'processed_subsets'
prot_param_dir = subset_dir / 'prot_param'
# Make the dir if we don't have it
prot_param_dir.mkdir(parents=True, exist_ok=True)

### Reading in data

In [ ]:
# Read annotated dataset
annotated_data = 'uniprotkb_AND_model_organism_9606_2025_09_09_annotated.parquet.gz'
df = pl.read_parquet(data_dir / annotated_data)

### Thresholding

In [ ]:
# Thresholds - must be separate clusters in this mmseqs category to be put into test set
test_threshold = 20 #0.20
val_threshold = 50 # 40% identity minimum

### Unpacking column data

In [ ]:
# Unpack the 'secondary structure fraction' calculations
secondary_structures = ['helix', 'turn', 'sheet']
ss_cols = []
ss_col_names = []
for i,s in enumerate(secondary_structures):
    aka = f"fraction_{s}_aas"
    ss_col_names.append(aka)
    ss_cols.append(pl.col('secondary_structure_fraction').list[i].alias(aka))

# Unpack the percent_amino_acids calculations
# Do a quick ProteinAnalysis to get the order of dictionary items correct
X = ProteinAnalysis(df['sequence'][0])
aa_list = list(X.amino_acids_percent.keys())
aa_cols = []
aa_col_names = []
for aa_index in range(len(aa_list)):
    aka = f"percent_{aa_list[aa_index]}"
    aa_col_names.append(aka)
    aa_cols.append(pl.col('amino_acid_percent').struct[aa_index].alias(aka))
# Add the columns to the dataframe
df = df.with_columns(ss_cols + aa_cols)

In [ ]:
# Select desired columns for protein-level prediction
target_cols = {
    'id_cols': ['id', 'sequence'],
    'mmseqs_cols': [c for c in df.columns if ('mmseqs' in c)],
    'size_cols': ['Length', 'Mass'],
    'composition_cols': aa_col_names,
    'structure_composition_cols': ss_col_names,
    'flex_cols': ['instability_index'],
    'hydrophobicity_cols': ['gravy'],
    'charge_cols': ['isoelectric_point', 'charge_at_ph4_7', 'charge_at_ph7_2', 'charge_at_ph8']
}
all_target_cols = []
[all_target_cols.extend(v) for v in target_cols.values()]
df = df.select(all_target_cols)
# Reformat columns
df.columns = [c.lower() for c in df.columns] 

In [ ]:
df

# Split the dataset different ways

In [ ]:
num_splits = 3
df_splits = {}
# Do a clustered split on similarity - we mostly care about not having very similar proteins in the training and test sets
for i in range(num_splits):
    split_df = pl.concat(hierarchical_clustered_split(df, 
                val_threshold = f'mmseqs_0.{val_threshold}',
                test_threshold = f'mmseqs_0.{test_threshold}',
                val_ratio = 0.15,
                test_ratio = 0.15,
                seed = SEED + i
    ))
    df_splits[i] = split_df

df = df_splits[0]

In [ ]:
df

### Sanity checks to make sure that we're making different splits each time

In [ ]:
def get_overlap_matrix(df_splits, idx_a=0, idx_b=1):
    df_a = df_splits[idx_a]
    df_b = df_splits[idx_b]
    
    categories = ['train', 'val', 'test']
    matrix = []

    for cat_a in categories:
        row = []
        # Get IDs for category in first split
        ids_a = set(df_a.filter(pl.col("split") == cat_a)["id"])
        
        for cat_b in categories:
            # Get IDs for category in second split
            ids_b = set(df_b.filter(pl.col("split") == cat_b)["id"])
            
            # Calculate intersection size
            overlap = len(ids_a.intersection(ids_b))
            row.append(overlap)
        matrix.append(row)

    return pd.DataFrame(matrix, index=categories, columns=categories)

# Example: Compare Split 0 against Split 1
overlap_df = get_overlap_matrix(df_splits, 0, 1)
print("overlap 0,1")
print(overlap_df)
overlap_df = get_overlap_matrix(df_splits, 0, 2)
print("overlap 0,2")
print(overlap_df)
overlap_df = get_overlap_matrix(df_splits, 1, 2)
print("overlap 1,2")
print(overlap_df)

def check_split_overlaps(df_splits):
    for i, df in df_splits.items():
        print(f"--- Analysis for Split {i} ---")
        
        # 1. Extract IDs for each group
        train_ids = set(df.filter(pl.col("split") == "train")["id"])
        val_ids   = set(df.filter(pl.col("split") == "val")["id"])
        test_ids  = set(df.filter(pl.col("split") == "test")["id"])
        
        # 2. Calculate Intersections
        train_val_overlap = train_ids.intersection(val_ids)
        train_test_overlap = train_ids.intersection(test_ids)
        val_test_overlap = val_ids.intersection(test_ids)
        
        # 3. Display Results
        print(f"Train size: {len(train_ids)} | Val size: {len(val_ids)} | Test size: {len(test_ids)}")
        
        if not (train_val_overlap or train_test_overlap or val_test_overlap):
            print("No overlaps detected. Splits are clean.")
        else:
            print("Overlaps detected!")
            print(f"  - Train/Val overlap: {len(train_val_overlap)}")
            print(f"  - Train/Test overlap: {len(train_test_overlap)}")
            print(f"  - Val/Test overlap: {len(val_test_overlap)}")
        print("\n")

# Run the check
check_split_overlaps(df_splits)

We don't seem to have data leakage issues, and we are getting different ids in our train:val:test splits. It's not huuuugely different like a true cross-validation split would be, but the main goal is to ensure more variability in train:testing without invalidating previous results (which were done on split i=0). We don't want to do true cross-validation from the get-go because we want to keep most of the data for training, some of the datasets are quite small and we need enough samples to learn from.

### Checking how much difference there is between train, val and test

In [ ]:
categories_to_plot = [c for c in df.columns if (df[c].dtype != pl.String)]
# Determine the number of subplots (rows and columns)
num_cats = len(categories_to_plot)
# Use 2 columns for a clean layout, adjust as needed
cols = 3
rows = (num_cats + cols - 1) // cols

# Create the figure with subplots
fig = make_subplots(
    rows=rows, 
    cols=cols, 
    # Use 'category name' as the subplot title
    subplot_titles=categories_to_plot
)

# Define the groups and their labels
group_labels = ['train', 'val', 'test']

# Iterate through the categories and add a distplot to each subplot
for i, cat in enumerate(categories_to_plot):
    # Calculate the row and column for the current subplot
    row = (i // cols) + 1
    col = (i % cols) + 1

    # Prepare the data for the current category
    hist_data = [
        df.filter(pl.col('split') == 'train')[cat].to_numpy(),
        df.filter(pl.col('split') == 'val')[cat].to_numpy(),
        df.filter(pl.col('split') == 'test')[cat].to_numpy(),
    ]

    # Create the distplot (this returns a figure with multiple traces)
    distplot_fig = ff.create_distplot(
        hist_data, 
        group_labels, 
        show_hist=False, 
        show_rug=False # Optional: for cleaner plot
    )
    
    # Add each trace (the KDE lines) from the distplot to the subplot
    for trace in distplot_fig['data']:
        fig.add_trace(
            trace,
            row=row,
            col=col
        )
    
# Update the overall layout
fig.update_layout(
    height=300 * rows, # Adjust height based on number of rows
    title_text="<b>Distribution of Categories by Split</b>",
    # Manually add a legend based on the last distplot's traces
    showlegend=True,
)

# To ensure the legend only shows once, we can use a trick:
# Go through all traces and hide the legend for all but the first set.
# The number of traces for one subplot is len(group_labels).
for i, trace in enumerate(fig.data):
    if i >= len(group_labels):
        trace.showlegend = False

fig.show()


For all properties, the train:val:test split doesn't seem to shift the distribution.

### Checking correlation between target variables

In [ ]:
check_correlation_plotly(df, color_map='viridis', height=1000, width = 1000)

&rarr; there are correlations, many make sense (charge at different pH, isoelectric point; mass and length; gravy highly associated with valines and leucines and met, ile, phe)

In [ ]:
df

In [ ]:
# Save each split
for i, df in df_splits.items():
    if i !=0:
        appendix = '_split' + str(i)
    else:
        appendix = ''
    # Make a dataset that's a subset for the checkpoints experiments - it has a max context length of 512 tokens.
    df_512 = df.filter(pl.col('sequence').str.len_chars() <= 512)
    print(df_512['split'].value_counts(normalize=True))
    df_512.write_parquet(prot_param_dir /f"h_sapiens_proteome_prot_param_protein_level_clustersplit_{test_threshold}_{val_threshold}_512_cutoff{appendix}.parquet.gz")
    # There's already a roughly 70:15:15 split, let's save
    #Save the df
    df.write_parquet(prot_param_dir / f"h_sapiens_proteome_prot_param_protein_level_clustersplit_{test_threshold}_{val_threshold}{appendix}.parquet.gz")